# Document token contributions

Global token contributions and comparisons of quotation marks, emphasis, and capitalisation. Run the SHAP notebook first.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Global token contributions

In [ ]:
import pandas as pd
import re
import os
import spacy
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm')
csv_path = str(paths.document_run() / 'general/shap_output_all_folds.csv')
output_dir = str(paths.document_run() / 'general/plots/')
max_token_length = 2000
min_count = 3


def is_clean_word(token):
    return re.fullmatch('[A-Za-z]+', token) is not None


def lemmatize_token(token):
    doc = nlp(token)
    return doc[0].lemma_
df = pd.read_csv(csv_path)
if df.empty:
    raise ValueError(f"The CSV file '{csv_path}' is empty.")
print(f' Loaded {len(df)} rows from {csv_path}')
if 'text_id' in df.columns:
    print(f" Distinct text_id count: {df['text_id'].nunique()}")
df = df[df['shap_Ref'].abs() > 1e-06]
df['token'] = df['token'].astype(str)
df['token_lower'] = df['token'].str.lower()
df = df[(df['token'].str.len() <= max_token_length) & df['token_lower'].apply(is_clean_word) & ~df['token_lower'].isin(stop_words)]
if df.empty:
    raise ValueError('No tokens remaining after filtering.')
df['lemma'] = df['token_lower'].apply(lemmatize_token)
df['lemma'] = df['lemma'].replace({'dicken': 'dickens'})
lemma_counts = df['lemma'].value_counts()
valid_lemmas = lemma_counts[lemma_counts >= min_count].index
df = df[df['lemma'].isin(valid_lemmas)]
shap_sum = df.groupby('lemma')['shap_Ref'].sum().reset_index()
shap_mean = df.groupby('lemma')['shap_Ref'].mean().reset_index()
data_dict = {'sum': shap_sum, 'mean': shap_mean}
print(f' Data ready: {len(shap_sum)} lemmas (summed), {len(shap_mean)} (mean)')


## Positive and negative token distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

shap_sum = data_dict['sum']
shap_mean = data_dict['mean']


def plot_single(shap_df, mode, kind, output_path, n=15):
    sns.set_style('whitegrid')
    if kind == 'positive':
        top = shap_df[shap_df['shap_Ref'] > 0].sort_values('shap_Ref', ascending=False).head(n)
        color = 'lightcoral'
    else:
        top = shap_df[shap_df['shap_Ref'] < 0].sort_values('shap_Ref', ascending=True).head(n)
        color = '#4682B4'
    plt.figure(figsize=(7, 6))
    ax = sns.barplot(y='lemma', x='shap_Ref', data=top, color=color)
    ax.set_xlabel(f'{mode} SHAP Value', fontsize=13)
    ax.set_ylabel('Token (lemma)', fontsize=13)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=17)
    if kind == 'positive':
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')
        ax.tick_params(axis='y', length=0)
    for i, v in enumerate(top['shap_Ref']):
        if kind == 'positive':
            ax.text(v - 0.002, i, f'{v:.2f}', va='center', ha='right', color='white', fontsize=14)
        else:
            ax.text(v + 0.002, i, f'{v:.2f}', va='center', ha='left', color='white', fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: {output_path}')
os.makedirs(output_dir, exist_ok=True)
plot_single(shap_sum, 'Summed', 'positive', os.path.join(output_dir, 'shap_sum_positive.png'))
plot_single(shap_sum, 'Summed', 'negative', os.path.join(output_dir, 'shap_sum_negative.png'))
plot_single(shap_mean, 'Mean', 'positive', os.path.join(output_dir, 'shap_mean_positive.png'))
plot_single(shap_mean, 'Mean', 'negative', os.path.join(output_dir, 'shap_mean_negative.png'))


## Quotation and emphasis markers

In [ ]:
import pandas as pd

csv_path = str(paths.document_run() / 'general/shap_output_all_folds.csv')
df = pd.read_csv(csv_path)
quote_tokens = ['"', "'", '“', '”', '‘', '’', '„', '«', '»', '‹', '›']


def process_quote_markers(df, marker_tokens, output_path):
    inside, outside = ([], [])
    for text_id, group in df.groupby('text_id'):
        tokens = group['token'].tolist()
        shap_values = group['shap_Ref'].tolist()
        marked = [False] * len(tokens)
        stack = []
        for i, token in enumerate(tokens):
            if token in marker_tokens:
                if stack and stack[-1] == token:
                    stack.pop()
                else:
                    stack.append(token)
            elif stack:
                marked[i] = True
        for i, token in enumerate(tokens):
            if not isinstance(token, str) or token.strip() == '' or token in marker_tokens:
                continue
            entry = (token.lower(), shap_values[i])
            (inside if marked[i] else outside).append(entry)
    df_inside = pd.DataFrame(inside, columns=['token', 'shap_Ref'])
    df_outside = pd.DataFrame(outside, columns=['token', 'shap_Ref'])
    inside_stats = df_inside.groupby('token')['shap_Ref'].agg(['mean', 'count']).rename(
        columns={'mean': 'inside_mean', 'count': 'count_inside'},
    )
    outside_stats = df_outside.groupby('token')['shap_Ref'].agg(['mean', 'count']).rename(
        columns={'mean': 'outside_mean', 'count': 'count_outside'},
    )
    comparison = inside_stats.join(outside_stats, how='inner')
    comparison['diff'] = comparison['inside_mean'] - comparison['outside_mean']
    comparison = comparison.sort_values(by='diff', ascending=False)
    comparison.to_csv(output_path)
    print(f' Quotes comparison CSV saved to: {output_path}')
    global_mean_inside = df_inside['shap_Ref'].mean()
    global_mean_outside = df_outside['shap_Ref'].mean()
    delta = global_mean_inside - global_mean_outside
    print(
        f' Quotes: Δ SHAP = {delta:.6f} (inside: {global_mean_inside:.6f}, outside: {global_mean_outside:.6f})',
    )


def process_double_asterisk(df, output_path):
    inside, outside = ([], [])
    for text_id, group in df.groupby('text_id'):
        tokens = group['token'].tolist()
        shap_values = group['shap_Ref'].tolist()
        marked = [False] * len(tokens)
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == '*' and tokens[i + 1] == '*':
                start = i
                i += 2
                while i < len(tokens) - 1:
                    if tokens[i] == '*' and tokens[i + 1] == '*':
                        for j in range(start + 2, i):
                            marked[j] = True
                        i += 2
                        break
                    i += 1
            else:
                i += 1
        for i, token in enumerate(tokens):
            if not isinstance(token, str) or token.strip() == '' or token == '*':
                continue
            entry = (token.lower(), shap_values[i])
            (inside if marked[i] else outside).append(entry)
    df_inside = pd.DataFrame(inside, columns=['token', 'shap_Ref'])
    df_outside = pd.DataFrame(outside, columns=['token', 'shap_Ref'])
    inside_stats = df_inside.groupby('token')['shap_Ref'].agg(['mean', 'count']).rename(
        columns={'mean': 'inside_mean', 'count': 'count_inside'},
    )
    outside_stats = df_outside.groupby('token')['shap_Ref'].agg(['mean', 'count']).rename(
        columns={'mean': 'outside_mean', 'count': 'count_outside'},
    )
    comparison = inside_stats.join(outside_stats, how='inner')
    comparison['diff'] = comparison['inside_mean'] - comparison['outside_mean']
    comparison = comparison.sort_values(by='diff', ascending=False)
    comparison.to_csv(output_path)
    print(f' Asterisk comparison CSV saved to: {output_path}')
    global_mean_inside = df_inside['shap_Ref'].mean()
    global_mean_outside = df_outside['shap_Ref'].mean()
    delta = global_mean_inside - global_mean_outside
    print(
        f' Asterisks: Δ SHAP = {delta:.6f} (inside: {global_mean_inside:.6f}, outside: {global_mean_outside:.6f})',
    )
output_path_quotes = str(paths.document_run() / 'general' / 'shap_comparison_quotes.csv')
output_path_asterisks = str(paths.document_run() / 'general' / 'shap_comparison_asterisks.csv')
process_quote_markers(df, quote_tokens, output_path_quotes)
process_double_asterisk(df, output_path_asterisks)


## Publication terms inside and outside quotation marks

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

selected_tokens = ['publish', 'chapter', 'book', 'printing']
data = pd.read_csv(paths.document_run() / 'general' / 'shap_comparison_quotes.csv')
data = data.set_index('token').reindex(selected_tokens).dropna().reset_index()
df_plot = pd.DataFrame(data)
labels = df_plot['token']
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
color_inside = '#4D4D4D'
color_outside = '#B3B3B3'
bars1 = ax.bar(
    [i - width / 2 for i in x],
    df_plot['inside_mean'],
    width,
    label='Inside quotes',
    color=color_inside,
)
bars2 = ax.bar(
    [i + width / 2 for i in x],
    df_plot['outside_mean'],
    width,
    label='Outside quotes',
    color=color_outside,
)
ax.set_ylabel('Mean SHAP Value')
ax.set_xlabel('Token')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    h1 = bar1.get_height()
    ax.annotate(
        f"{h1:.3f}\n({df_plot['count_inside'].iloc[i]})",
        xy=(bar1.get_x() + bar1.get_width() / 2, h1),
        xytext=(0, 3),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
    h2 = bar2.get_height()
    token = df_plot['token'].iloc[i]
    if token == 'printing':
        y_offset = 20
        va_align = 'top'
    elif h2 < 0:
        y_offset = -10
        va_align = 'top'
    else:
        y_offset = 3
        va_align = 'bottom'
    ax.annotate(
        f"{h2:.3f}\n({df_plot['count_outside'].iloc[i]})",
        xy=(bar2.get_x() + bar2.get_width() / 2, h2),
        xytext=(3, y_offset),
        textcoords='offset points',
        ha='center',
        va=va_align,
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
plt.tight_layout()
output_path = str(paths.FIGURES_DIR / 'shap_quotes_signalwords.png')
plt.savefig(output_path, dpi=300)
print(f' Plot saved to: {output_path}')
plt.show()


## Character names inside and outside emphasis

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

selected_tokens = ['wegg', 'boffin', 'captain']
data = pd.read_csv(paths.document_run() / 'general' / 'shap_comparison_asterisks.csv')
data = data.set_index('token').reindex(selected_tokens).dropna().reset_index()
df_plot = pd.DataFrame(data)
labels = df_plot['token']
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
color_inside = '#4D4D4D'
color_outside = '#B3B3B3'
bars1 = ax.bar(
    [i - width / 2 for i in x],
    df_plot['inside_mean'],
    width,
    label='Inside asterisks',
    color=color_inside,
)
bars2 = ax.bar(
    [i + width / 2 for i in x],
    df_plot['outside_mean'],
    width,
    label='Outside asterisks',
    color=color_outside,
)
ax.set_ylabel('Mean SHAP Value')
ax.set_xlabel('Character Token')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    h1 = bar1.get_height()
    ax.annotate(
        f"{h1:.3f}\n({df_plot['count_inside'].iloc[i]})",
        xy=(bar1.get_x() + bar1.get_width() / 2, h1),
        xytext=(0, 3),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
    h2 = bar2.get_height()
    token = df_plot['token'].iloc[i]
    if token == 'captain':
        y_offset = 25
        va_align = 'top'
    elif h2 < 0:
        y_offset = -10
        va_align = 'top'
    else:
        y_offset = 3
        va_align = 'bottom'
    ax.annotate(
        f"{h2:.3f}\n({df_plot['count_outside'].iloc[i]})",
        xy=(bar2.get_x() + bar2.get_width() / 2, h2),
        xytext=(3, y_offset),
        textcoords='offset points',
        ha='center',
        va=va_align,
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
plt.tight_layout()
output_path = str(paths.FIGURES_DIR / 'shap_asterisks_signalwords.png')
plt.savefig(output_path, dpi=300)
print(f' Plot saved to: {output_path}')
plt.show()


## Capitalisation table

In [ ]:
import pandas as pd

csv_path = str(paths.document_run() / 'general/shap_output_all_folds.csv')
df = pd.read_csv(csv_path)
df = df[df['token'].apply(lambda x: isinstance(x, str) and x.isalpha())]
df_caps = df[df['token'].str.match('^[A-Z][a-z]+$')].copy()
df_caps['token_lower'] = df_caps['token'].str.lower()
avg_shap_caps = df_caps['shap_Ref'].mean()
print(f' Average SHAP value of capitalised tokens: {avg_shap_caps:.6f}')
df_lower = df[df['token'].str.match('^[a-z]+$')].copy()
df_lower_grouped = df_lower.groupby('token')['shap_Ref'].mean()
df_lower_counts = df_lower['token'].value_counts()
df_caps_grouped = df_caps.groupby('token_lower')['shap_Ref'].mean()
df_caps_counts = df_caps['token_lower'].value_counts()
comparison_data = []
for token, cap_mean in df_caps_grouped.items():
    if token in df_lower_grouped.index:
        lower_mean = df_lower_grouped[token]
        cap_count = df_caps_counts.get(token, 0)
        lower_count = df_lower_counts.get(token, 0)
        diff = cap_mean - lower_mean
        comparison_data.append((token, cap_mean, lower_mean, diff, cap_count, lower_count))
df_comparison = pd.DataFrame(
    comparison_data,
    columns=['token', 'shap_cap', 'shap_lower', 'diff', 'count_cap', 'count_lower'],
).sort_values(
    by='diff',
    ascending=False,
)
output_path = str(paths.document_run() / 'general' / 'shap_comparison_capitalisation.csv')
df_comparison.to_csv(output_path, index=False)
print(f' Capitalisation comparison CSV saved to: {output_path}')
print(f' Tokens compared: {len(df_comparison)}')


## Capitalisation of title words

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

selected_tokens = [
    'expectations',
    'twist',
    'little',
    'great',
    'curiosity',
    'shop',
    'mutual',
    'hard',
    'papers',
    'friend',
]
data = pd.read_csv(paths.document_run() / 'general' / 'shap_comparison_capitalisation.csv')
data = data.set_index('token').reindex(selected_tokens).dropna().reset_index()
df_plot = pd.DataFrame(data)
labels = df_plot['token']
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
color_cap = '#4D4D4D'
color_lower = '#B3B3B3'
bars1 = ax.bar([i - width / 2 for i in x], df_plot['shap_cap'], width, label='Capitalised', color=color_cap)
bars2 = ax.bar(
    [i + width / 2 for i in x],
    df_plot['shap_lower'],
    width,
    label='Lower-case',
    color=color_lower,
)
ax.set_ylabel('Mean SHAP Value')
ax.set_xlabel('Title Word')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend()
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    h1 = bar1.get_height()
    ax.annotate(
        f"{h1:.3f}\n({df_plot['count_cap'].iloc[i]})",
        xy=(bar1.get_x() + bar1.get_width() / 2, h1),
        xytext=(0, 3),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
    h2 = bar2.get_height()
    if df_plot['token'].iloc[i] == 'papers':
        y_offset = 45
        va_align = 'top'
    elif df_plot['token'].iloc[i] == 'twist':
        y_offset = 30
        va_align = 'top'
    else:
        y_offset = 3
        va_align = 'bottom'
    ax.annotate(
        f"{h2:.3f}\n({df_plot['count_lower'].iloc[i]})",
        xy=(bar2.get_x() + bar2.get_width() / 2, h2),
        xytext=(3, y_offset),
        textcoords='offset points',
        ha='center',
        va=va_align,
        fontsize=9,
        color='black',
        linespacing=1.2,
    )
plt.tight_layout()
output_path = str(paths.FIGURES_DIR / 'shap_capitalisation_titles_top10.png')
plt.savefig(output_path, dpi=300)
print(f' Plot saved to: {output_path}')
plt.show()


## Individual letter

Set `TEXT_ID` to a letter ID from the attribution table. Token labels are masked by default.

In [ ]:
from shared.visualization import plot_split_shap_for_letter, render_token_attributions

TEXT_ID = None
SHOW_TEXT = False
if TEXT_ID is not None:
    plot_split_shap_for_letter(
        paths.document_run() / 'general' / 'shap_output_all_folds.csv',
        TEXT_ID,
        paths.FIGURES_DIR / 'letter_token_contributions.png',
        n=5,
        show_text=SHOW_TEXT,
    )
    render_token_attributions(
        paths.document_run() / 'general' / 'shap_output_all_folds.csv',
        TEXT_ID,
        paths.FIGURES_DIR / 'letter_token_attributions.html',
        show_text=SHOW_TEXT,
    )
